# Inter-rater agreement

## Issuebench

In [1]:
import json
import hashlib

import pandas as pd

from collections import defaultdict
from sklearn.metrics import cohen_kappa_score
from llm_audit import BASE_DIR


def hash_text(text: str) -> str:
    sha256_hash = hashlib.sha256()
    sha256_hash.update(text.encode("utf-8"))
    hashed_text = sha256_hash.hexdigest()
    return hashed_text


variables = ["agression", "submission", "conventionalism", "refusal"]
rater_labels = ["Andreas", "Max"]
rater = defaultdict(list)

dat = []
with open(
    BASE_DIR / "eval" / "data" / "human_annotations" / "issuebench" / "andreas_labels.json",
    "r",
) as f:
    dat.append(("Andreas", json.load(f)))
with open(
    BASE_DIR / "eval" / "data" / "human_annotations" / "issuebench" / "max_labels.json",
    "r",
) as f:
    dat.append(("Max", json.load(f)))

for rater_label, data in dat:
    for d in data:
        task_key = hash_text(d["data"]["judge_prompt"])
        annotations = d["annotations"][0]
        # id = annotations["id"] # = annotations["task"]
        task_rater_dict = {}
        if len(annotations["result"]) < len(variables):
            continue  # human label error (missing value)
        for r in annotations["result"]:
            variable = r["from_name"]
            label = 1 if r["value"]["choices"][0] == "Yes" else 0
            task_rater_dict[variable] = label
        rater[task_key].append({"rater_label": rater_label, "data": task_rater_dict})

data = defaultdict(list)
for task_key, rater_dicts in rater.items():
    if len(rater_dicts) == len(rater_labels):
        for rater_dict in rater_dicts:
            rater_label = rater_dict["rater_label"]
            rater_data = rater_dict["data"]
            for variable, label in rater_data.items():
                data[f"{variable}_{rater_label}"].append(label)

df = pd.DataFrame(data)
# df.to_csv("rater-comp.csv", encoding='utf-8', index=False, header=True)

rater_1 = "Andreas"
rater_2 = "Max"
for variable in variables:
    print(
        variable,
        cohen_kappa_score(df[f"{variable}_{rater_1}"], df[f"{variable}_{rater_2}"]),
    )
joint_var_by_rater = defaultdict(list)
for rater in [rater_1, rater_2]:
    for variable in variables:
        joint_var_by_rater[rater].extend(df[f"{variable}_{rater}"])
print("total", cohen_kappa_score(joint_var_by_rater[rater_1], joint_var_by_rater[rater_2]))

agression 0.2781774580335731
submission 0.13256484149855907
conventionalism 0.42152466367713004
refusal 1.0
total 0.4083247334021328
